In [1]:
# imports
import os
import lightgbm as lgb
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import TimeSeriesSplit
from datetime import date, timedelta
import yfinance as yf
import shap
import matplotlib.pyplot as plt
import streamlit as st
import yfinance as yf
import plotly.graph_objects as go
from scipy import stats
import statsmodels.api as sm

os.makedirs('../../../outputs/v3', exist_ok=True)

### Training

In [2]:
def make_splits(df, n_splits=5, target_col='target', gap_days=21):
    """
    Create walk-forward splits from the merged DataFrame.
    gap_days: trading days between train end and test start (avoids lookahead).
    """
    # Work on unique dates to split time, not rows
    dates = df.index.get_level_values('Date').unique().sort_values()
    tss = TimeSeriesSplit(n_splits=n_splits, gap=gap_days)
    
    splits = []
    for train_idx, test_idx in tss.split(dates):
        train_dates = dates[train_idx]
        test_dates  = dates[test_idx]
        
        train_df = df[df.index.get_level_values('Date').isin(train_dates)]
        test_df  = df[df.index.get_level_values('Date').isin(test_dates)]
        
        splits.append((train_df, test_df))
    
    return splits

### v3 Design (fixed before further results)
1. **Target:** 1 if the stock's next-5-day return beats the median of its own GICS sector that day, so the model can't win by betting on one sector.
2. **Features:** each feature becomes its percentile rank across stocks on the same date (inside `train_evaluate`), so thresholds learned in one year carry over to another.
3. **Baseline:** price features (volatility, short-term reversal, momentum), not volatility alone.
4. **Holdout:** 2026 is kept out of everything above and used once, in the last section, after the design is final.

In [ ]:
HOLDOUT_START = '2026-01-01'   # 2026 is never used for training or tuning

sectors = pd.read_csv('../../../outputs/v3/data/sp500_tickers.csv').set_index('Symbol')['GICS Sector']

def set_sector_target(d):
    """Target = 1 if fwd_5d beats the median of the stock's GICS sector on that date."""
    d = d.copy()
    d['Sector'] = d.index.get_level_values('Ticker').map(sectors).fillna('Unknown')
    sector_median = d.groupby([d.index.get_level_values('Date'), d['Sector']])['fwd_5d'].transform('median')
    d['Target'] = (d['fwd_5d'] > sector_median).astype(int)
    return d

def rank_by_date(X):
    """Percentile rank of each feature across stocks on the same date (uses no other dates)."""
    return X.groupby(level='Date').rank(pct=True)

price_cols = ['vol_5d', 'vol_20d', 'vol_spike',     # volatility
              'ret_1d', 'ret_5d', 'ret_20d',        # short-term reversal and momentum
              'rsi_14', 'pct_from_52wk_high']
text_cols = ['ai_demand_up_share', 'ai_demand_down_share', 'capex_up_share', 'capex_down_share',
             'competition_share', 'supply_limited_share', 'ai_demand_net', 'capex_net',
             'ai_tone', 'ai_sentence_ratio']                  # scripts/score_filings.py
news_cols = ['news_sentiment', 'news_count', 'pct_positive', 'eod_sentiment']
feat_cols = price_cols + text_cols + news_cols

In [ ]:
panel_all = pd.read_csv('../../../outputs/v3/data/ai_sector_features_df.csv', index_col=['Date', 'Ticker'], parse_dates=['Date'])
panel_all = set_sector_target(panel_all)
is_dev = panel_all.index.get_level_values('Date') < HOLDOUT_START
ai_sector_features_signals_df = panel_all[is_dev].copy()     # development sample, 2019-2025
holdout_df = panel_all[~is_dev].copy()                       # 2026 holdout
ai_sector_features_signals_df['regime'] = np.where(ai_sector_features_signals_df.index.get_level_values('Date') < '2023-01-01', 'pre_chatgpt', 'post_chatgpt')
splits = make_splits(ai_sector_features_signals_df, n_splits=5)
splits

In [4]:
# take fold's predictions and rescores them at a much finer resolution
# one IC number per trading day
def daily_ic_series(model, X_te, y_te, test_df):
    probs = model.predict_proba(X_te)[:, 1] # every row for predicting prob = 1 (outperform)
    temp = test_df.copy()
    temp['prob'] = probs
    temp['y'] = y_te.values

    daily_ic = temp.groupby(level='Date').apply(
        lambda g: np.corrcoef(g['prob'], g['y'])[0, 1] if g['y'].nunique() > 1 else np.nan
    )
    return daily_ic.dropna()

In [ ]:
# gap between training & validation 
PURGE_DAYS = 5  

def train_evaluate(splits, feat_cols):
    all_results = []
    all_daily_ic = []
    for fold_num, (train_df, test_df) in enumerate(splits):
        # use last 15% of training data for validation set, with a purge gap
        train_dates = train_df.index.get_level_values('Date').unique().sort_values()
        cut_idx = int(len(train_dates) * 0.85)
        cutoff = train_dates[cut_idx]
        fit_end = train_dates[cut_idx - PURGE_DAYS]                                   # CHANGED
        fit_df = train_df[train_df.index.get_level_values('Date') <= fit_end]         # CHANGED
        val_df = train_df[train_df.index.get_level_values('Date') > cutoff]

        X_fit, y_fit = rank_by_date(fit_df[feat_cols]), fit_df['Target']
        X_val, y_val = rank_by_date(val_df[feat_cols]), val_df['Target']
        X_te, y_te   = rank_by_date(test_df[feat_cols]), test_df['Target']

        asset_returns = test_df['fwd_5d'].values   # realized next-5-day return (ret_5d is the past 5 days)

        model = lgb.LGBMClassifier(
            n_estimators=600,
            learning_rate=0.05,
            num_leaves=31,
            min_child_samples=20,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight='balanced',
            reg_alpha=0.1,
            reg_lambda=0.1
        )
        model.fit(X_fit, y_fit,
                  eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False)])

        probs = model.predict_proba(X_te)[:,1]
        ic  = np.corrcoef(probs, y_te)[0,1]
        acc = accuracy_score(y_te, probs > 0.5)

        # volatile threshold
        vol_5d_thresh   = np.percentile(test_df['vol_5d'],   70)

        # regime IC — split test set by ChatGPT inflection point
        pre_mask = test_df.index.get_level_values('Date') < '2023-01-01' # before GPT
        post_mask = ~pre_mask # after release of GPT

        # compute correlation between model predictions and outcomes -- need at least 30 rows in that regime, or else we mark at nan
        ic_pre = np.corrcoef(probs[pre_mask], y_te[pre_mask])[0,1] if pre_mask.sum() > 30 else np.nan
        ic_post = np.corrcoef(probs[post_mask], y_te[post_mask])[0,1] if post_mask.sum() > 30 else np.nan

        # sharpe ratio — CHANGED: one portfolio return per date, non-overlapping 5d periods
        signals = np.where(probs > 0.52, 1, np.where(probs < 0.48, -1, 0))
        strat_returns = signals * asset_returns
        test_dates = test_df.index.get_level_values('Date')
        n_active = pd.Series(np.abs(signals), index=test_dates).groupby(level=0).sum()
        port = pd.Series(strat_returns, index=test_dates).groupby(level=0).sum()
        port = (port / n_active.replace(0, np.nan)).fillna(0)   # no positions -> flat day
        port = port.iloc[::5]                                   # non-overlapping 5d returns
        # no rf subtraction: long-short book is self-financing
        sharpe = port.mean() / port.std() * np.sqrt(252 / 5) if port.std() > 0 else np.nan

        # big moves — CHANGED: only score rows where a position was taken
        big = (np.abs(asset_returns) > 0.02) & (signals != 0)
        correct_big_moves = (np.mean(signals[big] == np.sign(asset_returns[big]))
                             if big.any() else np.nan)

        X_te_filled = X_te.fillna(0)   # used for the baseline ONLY
        baseline_ic = np.corrcoef(X_te_filled['vol_5d'], y_te)[0,1]
        baseline_preds = (X_te_filled['vol_5d'] > X_te_filled['vol_5d'].median()).astype(int)
        baseline_acc = accuracy_score(y_te, baseline_preds)
        baseline_ic_pre = np.corrcoef(X_te_filled['vol_5d'][pre_mask], y_te[pre_mask])[0,1] if pre_mask.sum() > 30 else np.nan
        baseline_ic_post = np.corrcoef(X_te_filled['vol_5d'][post_mask], y_te[post_mask])[0,1] if post_mask.sum() > 30 else np.nan

        all_results.append({
            'fold': fold_num+1,
            'IC': round(ic, 4),
            'accuracy': round(acc, 4),
            'sharpe': round(sharpe, 4),
            'Accuracy on >2% moves': round(correct_big_moves, 3),
            'baseline_IC': round(baseline_ic, 4),
            'baseline_acc': round(baseline_acc, 4),
            'IC_pre_chatgpt': round(ic_pre, 4) if not np.isnan(ic_pre) else np.nan,
            'IC_post_chatgpt': round(ic_post, 4) if not np.isnan(ic_post) else np.nan,
            'baseline_IC_pre': round(baseline_ic_pre, 4) if not np.isnan(baseline_ic_pre) else np.nan,
            'baseline_IC_post': round(baseline_ic_post, 4) if not np.isnan(baseline_ic_post) else np.nan,
            'delta_pre': round(ic_pre - baseline_ic_pre, 4) if not np.isnan(ic_pre) else np.nan,
            'delta_post': round(ic_post - baseline_ic_post, 4) if not np.isnan(ic_post) else np.nan,
        })

        print(f"Fold {fold_num+1}: IC={ic:.4f}, Acc={acc:.4f}, best_iter={model.best_iteration_}")
        print(f"Total test rows: {len(signals)}")
        print(f"Rows with signal != 0: {(signals != 0).sum()}")
        print(f"vol_5d threshold: {vol_5d_thresh:.4f}")
        print(f"probs distribution: min={probs.min():.3f}, max={probs.max():.3f}, mean={probs.mean():.3f}")
        print(" ")
        # CHANGED: removed X_te = X_te.fillna(0) and the duplicate baseline recompute
        print(f"Baseline IC (vol only): {baseline_ic:.4f}")
        print(f"Baseline accuracy: {baseline_acc:.4f}")

        # CHANGED: model now sees the same unfilled features it was evaluated on
        daily_ic = daily_ic_series(model, X_te, y_te, test_df)
        all_daily_ic.append(daily_ic)

    results_df = pd.DataFrame(all_results)
    full_daily_ic = pd.concat(all_daily_ic)

    # fold-level summary
    print(f"\n=== Fold-Level Summary ===")
    print(f"Mean IC:     {results_df['IC'].mean():.4f}")
    print(f"Median IC:   {results_df['IC'].median():.4f}")
    print(f"IC Std:      {results_df['IC'].std():.4f}")
    print(f"Fold ICIR (n=5, descriptive): {results_df['IC'].mean() / results_df['IC'].std():.4f}")
    print(f"Mean baseline IC: {results_df['baseline_IC'].mean():.4f}")                        # ADDED
    print(f"Folds beating baseline: {(results_df['IC'] > results_df['baseline_IC']).sum()}/{len(results_df)}")  # ADDED

    # daily IC summary
    print(f"\n=== Daily IC Summary ===")
    print(f"\nMean daily IC: {full_daily_ic.mean():.4f}")
    print(f"Daily IC Std:  {full_daily_ic.std():.4f}")
    print(f"Daily ICIR:    {full_daily_ic.mean() / full_daily_ic.std():.4f}")
    print(f"N days:        {len(full_daily_ic)}")

    return results_df, full_daily_ic, model, X_te, y_te

In [ ]:
df = pd.read_csv('../../../outputs/v3/data/ai_sector_features_df.csv', parse_dates=['Date'])
df = df[df['Date'] < HOLDOUT_START]                            # 2026 holdout stays out
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

# --- Check 1: no NaN targets silently labeled 0 ---
print("Rows with NaN fwd_5d:", df['fwd_5d'].isna().sum())       # want 0 after dropna
print("Last date per ticker:", df.groupby('Ticker')['Date'].max().max().date())

# --- Check 2: fwd_5d equals the sum of the NEXT 5 daily returns ---
g = df.groupby('Ticker')['ret_1d']
true_fwd = sum(g.shift(-k) for k in range(1, 6))                 # ret(t+1)+...+ret(t+5)
ok = np.isclose(df['fwd_5d'], true_fwd, equal_nan=True)
print(f"fwd_5d matches true forward window: {ok.mean():.1%}")    # want 100%

# --- Check 3: Target is 1 exactly when fwd_5d beats that date's median ---
med = df.groupby('Date')['fwd_5d'].transform('median')
rebuilt = (df['fwd_5d'] > med).astype(int)
print(f"Target matches rebuilt target: {(rebuilt == df['Target']).mean():.1%}")  # want 100%

# --- Check 4: class balance stays near 50/50 ---
print(f"Share of Target=1: {df['Target'].mean():.3f}")          # want ~0.49–0.50

In [ ]:
results_df, full_daily_ic, final_model, X_te, y_te = train_evaluate(splits, feat_cols)
results_df

In [8]:
results_df.to_csv('../../../outputs/v3/results_df.csv', index=False)
results_df.to_latex('../../../outputs/v3/results_table.tex', index=False, float_format='%.4f')

In [9]:
icir = results_df['IC'].mean() / results_df['IC'].std()
print(f"Median IC: {results_df['IC'].median():.4f}")
print(f'Mean IC: {results_df['IC'].mean():.4f}' )
print(f"IC Std:  {results_df['IC'].std():.4f}")
print(f"ICIR:    {icir:.4f}")

Median IC: 0.0138
Mean IC: 0.0111
IC Std:  0.0237
ICIR:    0.4682


In [ ]:
d = df.reset_index() if 'Date' not in df.columns else df.copy()
d['Date'] = pd.to_datetime(d['Date'])

for c in ['ai_sentence_ratio', 'ai_demand_up_share']:
    xs = d.groupby('Date')[c].agg(['mean', 'std'])
    xs = xs[xs['mean'] > 0]                      # skip days where every stock is 0 (CV undefined)
    xs['cv'] = xs['std'] / xs['mean']
    p = xs.index < '2023-01-01'
    print(f"{c:18s} mean pre/post: {xs['mean'][p].mean():.3f} / {xs['mean'][~p].mean():.3f}"
          f"  |  cross-sec CV pre/post: {xs['cv'][p].mean():.3f} / {xs['cv'][~p].mean():.3f}")

In [ ]:
splits = make_splits(ai_sector_features_signals_df.sort_index(), n_splits=5)              # same df and folds as your headline run

res_full, daily_full, *_ = train_evaluate(splits, feat_cols)     # price + text + news
res_base, daily_base, *_ = train_evaluate(splits, price_cols)    # price only

delta = (daily_full - daily_base).dropna().sort_index()
nw = sm.OLS(delta.values, np.ones(len(delta))).fit(cov_type='HAC', cov_kwds={'maxlags': 20})

print(f"Full model:  mean fold IC = {res_full['IC'].mean():.4f}, mean daily IC = {daily_full.mean():.4f}")
print(f"Price-only:  mean fold IC = {res_base['IC'].mean():.4f}, mean daily IC = {daily_base.mean():.4f}")
print(f"ΔIC (text + news contribution) = {delta.mean():.4f}, t = {nw.tvalues[0]:.2f}, "
      f"Newey-West p = {nw.pvalues[0]:.3f}, days = {len(delta)}")

### Feature-Group Comparison (2021–2025)
EODHD news is sparse before 2021 (~5–6k articles a year, vs 55k+ from 2021), so news is tested only on 2021–2025. Each feature group is compared with the price-only model on the same folds.

In [ ]:
NEWS_START = '2021-01-01'

df_2021 = ai_sector_features_signals_df[
    ai_sector_features_signals_df.index.get_level_values('Date') >= NEWS_START].sort_index()
splits_2021 = make_splits(df_2021, n_splits=5)

feature_sets = {
    'price only':          price_cols,
    'price + news':        price_cols + news_cols,
    'price + 8-K text':    price_cols + text_cols,
    'price + text + news': price_cols + text_cols + news_cols,
}
daily = {name: train_evaluate(splits_2021, cols)[1] for name, cols in feature_sets.items()}

rows = []
for name in feature_sets:
    row = {'features': name, 'daily_IC': daily[name].mean()}
    if name != 'price only':
        d = (daily[name] - daily['price only']).dropna()
        m = sm.OLS(d.values, np.ones(len(d))).fit(cov_type='HAC', cov_kwds={'maxlags': 20})
        row.update({'delta_vs_price': d.mean(), 't': m.tvalues[0], 'p': m.pvalues[0]})
    rows.append(row)

ablation_2021 = pd.DataFrame(rows)
print(ablation_2021.round(4).to_string(index=False))
ablation_2021.to_csv('../../../outputs/v3/ablation_2021.csv', index=False)

In [ ]:
importance = pd.Series(
    final_model.feature_importances_, 
    index=feat_cols
).sort_values(ascending=False)

print(importance)

In [13]:
feat_cols

['vol_5d',
 'vol_20d',
 'vol_spike',
 'gpu_mentions',
 'capex_up_score',
 'capex_net',
 'competitor_mentions',
 'ai_sentence_ratio',
 'nvidia_mentions',
 'amd_mentions',
 'intel_mentions']

In [14]:
importance.to_csv('../../../outputs/v3/feature_importance.csv')

### Regime Analysis

In [15]:
pre_ics = results_df['IC_pre_chatgpt'].dropna().values
post_ics = results_df['IC_post_chatgpt'].dropna().values
pre_deltas = results_df['delta_pre'].dropna().values
post_deltas = results_df['delta_post'].dropna().values

In [16]:
print("=== Regime Analysis ===")
print(f"Mean pre-ChatGPT IC:  {pre_ics.mean():.4f}")
print(f"Mean post-ChatGPT IC: {post_ics.mean():.4f}")
print(f"IC improvement:       {post_ics.mean() - pre_ics.mean():.4f}")
print(f"\nMean pre-ChatGPT delta:  {pre_deltas.mean():.4f}")
print(f"Mean post-ChatGPT delta: {post_deltas.mean():.4f}")


=== Regime Analysis ===
Mean pre-ChatGPT IC:  0.0175
Mean post-ChatGPT IC: 0.0097
IC improvement:       -0.0078

Mean pre-ChatGPT delta:  0.0258
Mean post-ChatGPT delta: 0.0039


In [17]:
# Test 1: Is post-ChatGPT IC higher than pre?
t1, p1 = stats.ttest_ind(post_ics, pre_ics, equal_var=False)
print(f"\nIC regime test (post vs pre): t={t1:.3f}, p={p1:.3f}")

# Test 2: Is the delta larger post-ChatGPT?
t2, p2 = stats.ttest_ind(post_deltas, pre_deltas, equal_var=False)
print(f"Delta regime test (post vs pre): t={t2:.3f}, p={p2:.3f}")

# Test 3: Is post-ChatGPT IC significantly above zero?
t3, p3 = stats.ttest_1samp(post_ics, 0)
print(f"Post-ChatGPT IC vs zero: t={t3:.3f}, p={p3:.3f}")

# Test 4: Is pre-ChatGPT IC significantly above zero?
t4, p4 = stats.ttest_1samp(pre_ics, 0)
print(f"Pre-ChatGPT IC vs zero: t={t4:.3f}, p={p4:.3f}")

print(f"\nRegime IC summary:")
print(results_df[['fold', 'IC', 'baseline_IC', 'IC_pre_chatgpt', 
                   'IC_post_chatgpt', 'delta_pre', 'delta_post']])


IC regime test (post vs pre): t=-0.382, p=0.734
Delta regime test (post vs pre): t=-0.783, p=0.504
Post-ChatGPT IC vs zero: t=1.664, p=0.238
Pre-ChatGPT IC vs zero: t=0.898, p=0.464

Regime IC summary:
   fold      IC  baseline_IC  IC_pre_chatgpt  IC_post_chatgpt  delta_pre  \
0     1 -0.0213       0.0017         -0.0213              NaN    -0.0229   
1     2  0.0399       0.0089          0.0399              NaN     0.0309   
2     3  0.0248       0.0043          0.0339           0.0172     0.0695   
3     4 -0.0018      -0.0001             NaN          -0.0018        NaN   
4     5  0.0138      -0.0060             NaN           0.0138        NaN   

   delta_post  
0         NaN  
1         NaN  
2     -0.0065  
3     -0.0017  
4      0.0199  


In [18]:
# Split by regime
pre_daily = full_daily_ic[full_daily_ic.index < '2023-01-01']
post_daily = full_daily_ic[full_daily_ic.index >= '2023-01-01']

print(f"Pre-ChatGPT: n={len(pre_daily)}, mean IC={pre_daily.mean():.4f}")
print(f"Post-ChatGPT: n={len(post_daily)}, mean IC={post_daily.mean():.4f}")

t_stat, p_value = stats.ttest_ind(post_daily, pre_daily, equal_var=False)
print(f"t={t_stat:.3f}, p={p_value:.4f}")

Pre-ChatGPT: n=713, mean IC=0.0088
Post-ChatGPT: n=747, mean IC=0.0077
t=-0.086, p=0.9318


In [19]:
# Newey-West Test

ic = pd.concat([pre_daily, post_daily])          # keep dates as index, in order
post = np.r_[np.zeros(len(pre_daily)), np.ones(len(post_daily))]

X = sm.add_constant(post)
res = sm.OLS(ic.values, X).fit(cov_type="HAC", cov_kwds={"maxlags": 20})
print(res.params, res.pvalues)   # [const, post]: pre mean, post-minus-pre

[ 0.00882058 -0.001142  ] [0.60879498 0.96201943]


In [20]:
# ---- Baseline daily IC: raw vol_5d vs Target, per date, same as train_evaluate ----
def baseline_daily_ic_series(test_df):
    temp = test_df[['vol_5d', 'Target']].copy()
    temp['vol_5d'] = temp['vol_5d'].fillna(0)          # matches X_te_filled in train_evaluate
    return temp.groupby(level='Date').apply(
        lambda g: np.corrcoef(g['vol_5d'], g['Target'])[0, 1]
        if g['Target'].nunique() > 1 and g['vol_5d'].nunique() > 1 else np.nan
    ).dropna()

baseline_daily_ic = pd.concat([baseline_daily_ic_series(te) for _, te in splits])

# ---- ΔIC = model − baseline, aligned on date ----
delta = (full_daily_ic - baseline_daily_ic).dropna().sort_index()
print(f"Days: model={len(full_daily_ic)}, baseline={len(baseline_daily_ic)}, delta={len(delta)}")
print(f"Mean daily IC: model={full_daily_ic.mean():.4f}, baseline={baseline_daily_ic.mean():.4f}, "
      f"delta={delta.mean():.4f}")

# ---- Regime tests on ΔIC ----
pre_d  = delta[delta.index <  '2023-01-01']
post_d = delta[delta.index >= '2023-01-01']

t_w, p_w = stats.ttest_ind(post_d, pre_d, equal_var=False)

d = pd.concat([pre_d, post_d])
X = sm.add_constant(np.r_[np.zeros(len(pre_d)), np.ones(len(post_d))])
res_d = sm.OLS(d.values, X).fit(cov_type="HAC", cov_kwds={"maxlags": 20})

print(f"\nn_pre={len(pre_d)}, n_post={len(post_d)}")
print(f"Mean ΔIC  pre={pre_d.mean():.4f}  post={post_d.mean():.4f}  diff={res_d.params[1]:.4f}")
print(f"Welch:       t={t_w:.3f}, p={p_w:.4f}")
print(f"Newey-West:  t={res_d.tvalues[1]:.3f}, p={res_d.pvalues[1]:.4f}, lags=20")

Days: model=1460, baseline=1460, delta=1460
Mean daily IC: model=0.0082, baseline=-0.0105, delta=0.0188

n_pre=713, n_post=747
Mean ΔIC  pre=0.0227  post=0.0150  diff=-0.0077
Welch:       t=-0.382, p=0.7024
Newey-West:  t=-0.216, p=0.8288, lags=20


### Leakage Test

In [ ]:
base = df.reset_index(drop=('Date' in df.columns)).copy()   # works whether Date is a column or the index
base['Date'] = pd.to_datetime(base['Date'])
base = base.set_index(['Date', 'Ticker']).sort_index(level=['Ticker', 'Date'])

def make_dataset(base, k=0):
    """Panel with a 5-day forward-return target shifted by k days of overlap.
    k=0 is the correct target (days t+1..t+5)."""
    d = base.copy()
    d['fwd_5d'] = d.groupby(level='Ticker')['ret_1d'].transform(
        lambda x: x.rolling(5).sum().shift(-5 + k))
    d = d.dropna(subset=['fwd_5d'])
    return set_sector_target(d).sort_index()

In [ ]:
d0 = make_dataset(base, k=0)
splits0 = make_splits(d0, n_splits=5)

# Main model (price + text + news) and price-only model on the SAME folds
results_df, full_daily_ic, final_model, X_te, y_te = train_evaluate(splits0, feat_cols)
res_base, daily_base, *_ = train_evaluate(splits0, price_cols)

# RQ1: do text and news add anything beyond price features?
delta = (full_daily_ic - daily_base).dropna().sort_index()
nw = sm.OLS(delta.values, np.ones(len(delta))).fit(cov_type='HAC', cov_kwds={'maxlags': 20})

print("\n=== MAIN RESULTS (k = 0) ===")
print(f"Full model:   mean fold IC = {results_df['IC'].mean():.4f}, "
      f"fold ICIR = {results_df['IC'].mean() / results_df['IC'].std():.4f}, "
      f"mean daily IC = {full_daily_ic.mean():.4f}")
print(f"Price-only:   mean fold IC = {res_base['IC'].mean():.4f}, mean daily IC = {daily_base.mean():.4f}")
print(f"Text + news ΔIC = {delta.mean():.4f}, t = {nw.tvalues[0]:.2f}, Newey-West p = {nw.pvalues[0]:.3f}, "
      f"days = {len(delta)}")

results_df.to_csv('../../../outputs/v3/main_results.csv', index=False)

In [23]:
results = []
for k in range(5):
    d = make_dataset(base, k)
    res, daily, *_ = train_evaluate(make_splits(d, n_splits=5), feat_cols)
    t = d[d.index.get_level_values('Date').isin(daily.index)]
    vol_daily = t.groupby(level='Date').apply(lambda g: g['vol_5d'].corr(g['Target']))
    results.append({'k': k, 'daily': daily, 'vol_daily': vol_daily,
                    'mean_IC': res['IC'].mean(), 'daily_IC': daily.mean(),
                    'folds_>_base': int((res['IC'] > res['baseline_IC']).sum())})
    print(f"k={k} done")

leak = pd.DataFrame([{c: r[c] for c in ['k', 'mean_IC', 'daily_IC', 'folds_>_base']} for r in results])
print(leak.round(4).to_string(index=False))
leak.to_csv('../../../outputs/v3/leakage_dose_response.csv', index=False)

[LightGBM] [Info] Number of positive: 1596, number of negative: 1824
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000510 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 811
[LightGBM] [Info] Number of data points in the train set: 3420, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Fold 1: IC=-0.0214, Acc=0.4671, best_iter=1
Total test rows: 4365
Rows with signal != 0: 0
vol_5d threshold: 0.0286
probs distribution: min=0.480, max=0.516, mean=0.500
 
Baseline IC (vol only): 0.0020
Baseline accuracy: 0.4852
[LightGBM] [Info] Number of positive: 3332, number of negative: 3808
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000405 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col

In [24]:
def nw(s):
    s = pd.Series(s).dropna()
    m = sm.OLS(s.values, np.ones(len(s))).fit(cov_type='HAC', cov_kwds={'maxlags': 20})
    return m.params[0], m.tvalues[0], m.pvalues[0]

model = pd.concat({r['k']: r['daily'] for r in results}, axis=1).dropna()
vol   = pd.concat({r['k']: r['vol_daily'] for r in results}, axis=1).reindex(model.index)

rows = []
for k in model.columns:
    ic, t, p = nw(model[k])                                            # Is IC > 0 at this k?
    p_inc = nw(model[k] - model[0])[2] if k else np.nan                # Is it higher than at k=0?
    p_vol = nw(model[k] - vol[k])[2]                                   # Does model beat raw vol_5d?
    rows.append({'k': k, 'IC': ic, 't': t, 'p': p, 'p_vs_k0': p_inc,
                 'vol_IC': vol[k].mean(), 'p_model_vs_vol': p_vol})
tests = pd.DataFrame(rows)
print(tests.round(4).to_string(index=False))

ks = np.array(model.columns, float); kc = ks - ks.mean()
slope = model.values @ kc / (kc ** 2).sum()                            # Does IC rise with k?
b, t2, p2 = nw(slope)
print(f"\nTrend: +{b:.4f} IC per day of overlap, t={t2:.2f}, p={p2:.3g}")
tests.to_csv('../../../outputs/v3/leakage_tests.csv', index=False)

 k     IC      t      p  p_vs_k0  vol_IC  p_model_vs_vol
 0 0.0147 1.3145 0.1887      NaN -0.0103          0.1448
 1 0.0338 3.7077 0.0002   0.0808  0.0017          0.0357
 2 0.0346 3.4591 0.0005   0.0923  0.0188          0.3264
 3 0.0479 5.1046 0.0000   0.0116  0.0351          0.3808
 4 0.0496 4.8281 0.0000   0.0118  0.0521          0.8791

Trend: +0.0084 IC per day of overlap, t=2.51, p=0.0122


### Data Visualization

Note: Some cells below are repeated from `data_cleaning.ipynb`.

In [25]:
tickers = pd.read_csv('../../../data/tickers.csv')
tickers = tickers.drop(columns=['Unnamed: 0'])

ai_tickers = {
    'semis':   ['NVDA','AMD','INTC','ASML','TSM','AVGO','MRVL','ARM','SMCI'], # semiconductors
    'hypers':  ['MSFT','GOOGL','AMZN','META'], # hyperscalar
    'pure':    ['PLTR','AI','SOUN'], # AI pure plays
    'infra':   ['DELL','ANET','HPE'] # infrastructure
}
all_tickers = [t for group in ai_tickers.values() for t in group]
all_tickers

['NVDA',
 'AMD',
 'INTC',
 'ASML',
 'TSM',
 'AVGO',
 'MRVL',
 'ARM',
 'SMCI',
 'MSFT',
 'GOOGL',
 'AMZN',
 'META',
 'PLTR',
 'AI',
 'SOUN',
 'DELL',
 'ANET',
 'HPE']

In [26]:
# SOXX: holds 30 largest semiconductor companies
# QQQ: holds 100 largest non-financial companies on the NASDAQ, heavily weighted towards big tech
def make_features(close: pd.Series,
                     volume: pd.Series,
                     soxx: pd.Series,
                     qqq: pd.Series) -> pd.DataFrame:
    df = pd.DataFrame(index=close.index)

    # Lag returns: percent change over x days
    for lag in [1, 3, 5, 10, 20]:
        df[f'ret_{lag}d'] = close.pct_change(lag)

    # Rolling volatility
    df['vol_5d']  = close.pct_change().rolling(5).std()
    df['vol_20d'] = close.pct_change().rolling(20).std()

    # RSI (14-day): Relative Strength Index; measures magnitude and recent price changes in stocks
    delta = close.pct_change()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    df['rsi_14'] = 100 - (100 / (1 + gain / loss))

    # How does the stock compare to SOXX and QQQ?
    df['rel_soxx'] = close.pct_change(20) - soxx.pct_change(20)
    df['rel_qqq']  = close.pct_change(20) - qqq.pct_change(20)

    # Volume spike: number of shares traded on a given day
    df['vol_spike'] = volume / volume.rolling(20).mean()

    # 52-week high proximity: the highest price traded at over the last 252 trading days (1 year)
    df['pct_from_52wk_high'] = close / close.rolling(252).max() - 1

    return df.dropna()

In [ ]:
end = date.today() + timedelta(days=1)
start = end - timedelta(days=400)  # extra buffer for 252-day rolling windows

prices_live = yf.download(
    tickers=all_tickers + ['SOXX', 'QQQ'],
    start=start,
    end=end,
    auto_adjust=True,
    progress=False
)

close_live = prices_live['Close']
volume_live = prices_live['Volume']

# Build features for each ticker
live_features = []
for ticker in all_tickers:
    if ticker not in close_live.columns:
        print(f"Missing: {ticker}")
        continue
    feats = make_features(
        close=close_live[ticker],
        volume=volume_live[ticker],
        soxx=close_live['SOXX'],
        qqq=close_live['QQQ']
    )
    feats['Ticker'] = ticker
    live_features.append(feats)

live_df = pd.concat(live_features).reset_index()
live_df = live_df.set_index(['Date', 'Ticker'])

# Get the most recent row per ticker (should be June 18)
latest = live_df.groupby('Ticker').tail(1)
print(f"Latest date: {latest.index.get_level_values('Date').unique()}")
print(f"Tickers: {latest.index.get_level_values('Ticker').tolist()}")

for col in feat_cols:
    if col not in latest.columns:
        latest[col] = 0
latest[feat_cols] = latest[feat_cols].fillna(0)

probs_today = final_model.predict_proba(latest[feat_cols].rank(pct=True))[:, 1]   # same ranking as training

ranking = pd.Series(probs_today, index=latest.index.get_level_values('Ticker'))
ranking = ranking.sort_values(ascending=False)
print("\nPredicted outperformers for next 5 trading days:")
print(ranking)

In [28]:
colors = ['#2ecc71' if p > 0.5 else '#e74c3c' for p in ranking.values[::-1]]

fig = go.Figure(go.Bar(
    x=ranking.values[::-1],
    y=ranking.index[::-1],
    orientation='h',
    marker_color=colors,
    text=[f'{p:.3f}' for p in ranking.values[::-1]],
    textposition='outside'
))

fig.add_vline(
    x=0.5,
    line_dash='dash',
    line_color='black',
    annotation_text='Decision threshold (0.5)',
    annotation_position='top'
)

fig.update_layout(
    title=f'AI Sector Stock Rankings — Next 5 Trading Days<br>{latest.index.get_level_values("Date").unique()[0].date()}',
    xaxis_title='Predicted Outperformance Probability',
    xaxis=dict(range=[0, 0.75]),
    height=600,
    width=900,
    plot_bgcolor='white',
    showlegend=False
)

fig.show()

### SHAP Plot

In [29]:
# Main model (k=0 target, same splits as cell 24); don't overwrite results_df
_, daily_ic, final_model, X_test, Y_te = train_evaluate(splits0, feat_cols)
explainer   = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_test)

[LightGBM] [Info] Number of positive: 1596, number of negative: 1824
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000408 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 811
[LightGBM] [Info] Number of data points in the train set: 3420, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Fold 1: IC=-0.0214, Acc=0.4671, best_iter=1
Total test rows: 4365
Rows with signal != 0: 0
vol_5d threshold: 0.0286
probs distribution: min=0.480, max=0.516, mean=0.500
 
Baseline IC (vol only): 0.0020
Baseline accuracy: 0.4852
[LightGBM] [Info] Number of positive: 3332, number of negative: 3808
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000555 seconds.
You can set `force_row_wise=true` to remove t

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/shap/explainers/_tree.py:583: UserWarning:

LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray



In [30]:
# Feature importance from the main (k=0) model
importance = pd.Series(final_model.feature_importances_, index=feat_cols).sort_values(ascending=False)
importance.to_csv('../../../outputs/v3/feature_importance.csv')
print(importance)

vol_20d                12
nvidia_mentions         4
vol_spike               3
capex_up_score          3
competitor_mentions     3
ai_sentence_ratio       3
amd_mentions            2
vol_5d                  0
gpu_mentions            0
capex_net               0
intel_mentions          0
dtype: int32


In [31]:
shap.summary_plot(shap_values, X_test, max_display=20, show=False)
plt.gcf().set_size_inches(4,6)
plt.savefig('../../../outputs/v3/shap_summary.png', dpi=150, bbox_inches='tight')
plt.close()

In [32]:
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[1000],
        base_values=explainer.expected_value,
        data=X_test.round(3).iloc[1000],
        feature_names=feat_cols
    ),
    max_display=12,
    show=False
)
plt.savefig('../../../outputs/v3/shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.close()

### Other Visualizations

In [33]:
main_res = pd.read_csv('../../../outputs/v3/main_results.csv')
n = len(main_res)
x = np.arange(n)
fold_labels = [f'Fold {i}' for i in range(1, n + 1)]
model_ics = main_res['IC'].values
baseline_ics = main_res['baseline_IC'].values

In [34]:
# IC by fold bar chart
plt.close('all')
fig, ax = plt.subplots(figsize=(10, 5))
width = 0.35
ax.bar(x - width/2, model_ics, width, label='Model IC', color='#2ecc71')
ax.bar(x + width/2, baseline_ics, width, label='Baseline IC (vol_5d)', color='#95a5a6')
ax.axhline(0, color='black', linewidth=0.8)           # ICs can now be negative
ax.set_xlabel('Fold')
ax.set_ylabel('Information Coefficient')
ax.set_title('Model IC vs Volatility-Only Baseline by Fold')
ax.set_xticks(x)
ax.set_xticklabels(fold_labels)
ax.legend()
plt.tight_layout()
plt.savefig('../../../outputs/v3/ic_by_fold.png', dpi=150, bbox_inches='tight')
plt.close(fig)

In [35]:
# Cumulative IC Over Time
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, model_ics, marker='o', color='#2ecc71', linewidth=2, label='Model IC')
ax.plot(x, baseline_ics, marker='s', color='#95a5a6', linewidth=2, linestyle='--',
        label='Baseline IC (vol_5d)')
ax.fill_between(x, model_ics, baseline_ics, alpha=0.2, color='#2ecc71')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(fold_labels)
ax.set_xlabel('Fold')
ax.set_ylabel('Information Coefficient')
ax.set_title('Model vs Baseline IC Across Walk-Forward Folds')
ax.legend()
plt.tight_layout()
plt.savefig('../../../outputs/v3/ic_over_folds.png', dpi=150, bbox_inches='tight')
plt.close(fig)

In [ ]:
leak = pd.read_csv('../../../outputs/v3/leakage_dose_response.csv')   # written by the leakage cell above

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(leak['k'], leak['daily_IC'], marker='o', linewidth=2, label='Mean daily IC')
ax.plot(leak['k'], leak['mean_IC'], marker='s', linewidth=2, linestyle='--', label='Mean fold IC')
ax.axhline(0, color='black', linewidth=0.8)

ax.set_xticks(leak['k'])
ax.set_xticklabels(['0\n(correct)', '1', '2', '3', '4\n(original bug)'])
ax.set_xlabel('Days of overlap between label window and feature date (k)')
ax.set_ylabel('Information Coefficient (IC)')
ax.set_title('IC increases with lookahead leakage')
ax.set_ylim(bottom=-0.005)
ax.legend()
plt.tight_layout()
plt.savefig('../../../outputs/v3/leakage_dose_response.png', dpi=300, bbox_inches='tight')
plt.show()

### Streamlit Dashboard


In [37]:
# Gain importance (% of total)
gain = pd.Series(final_model.booster_.feature_importance(importance_type='gain'),
                 index=feat_cols)
gain_pct = (gain / gain.sum() * 100).sort_values(ascending=False)

# Mean |SHAP| on the Fold 5 test set
explainer = shap.TreeExplainer(final_model)
sv = explainer.shap_values(X_test)
if isinstance(sv, list):          # older LightGBM/SHAP returns [class0, class1]
    sv = sv[1]
mean_abs_shap = pd.Series(np.abs(sv).mean(axis=0), index=feat_cols)

importance_table = pd.DataFrame({
    'gain_pct': gain_pct.round(1),
    'mean_abs_shap': mean_abs_shap.round(4),
}).sort_values('mean_abs_shap', ascending=False)
print(importance_table)

# Regenerate Figure 2 from the correct model
shap.summary_plot(sv, X_test, max_display=12, show=False)
plt.gcf().set_size_inches(4, 6)
plt.savefig('../../../outputs/v3/shap_summary.png', dpi=150, bbox_inches='tight')
plt.close()

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/shap/explainers/_tree.py:583: UserWarning:

LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray



                     gain_pct  mean_abs_shap
vol_20d                  32.9         0.0052
competitor_mentions      14.3         0.0032
nvidia_mentions          19.3         0.0020
vol_spike                 9.2         0.0018
ai_sentence_ratio         9.1         0.0010
amd_mentions              5.6         0.0007
capex_up_score            9.6         0.0001
capex_net                 0.0         0.0000
gpu_mentions              0.0         0.0000
intel_mentions            0.0         0.0000
vol_5d                    0.0         0.0000


### 2026 Holdout (run once, after the design is final)
Both models are trained on all of 2019–2025, leaving a 21-trading-day gap before 2026 as in the walk-forward folds, and scored once on 2026.

In [ ]:
dev_dates = ai_sector_features_signals_df.index.get_level_values('Date').unique().sort_values()
train_all = ai_sector_features_signals_df[ai_sector_features_signals_df.index.get_level_values('Date') <= dev_dates[-22]]
holdout_split = [(train_all, holdout_df)]

res_h_full, daily_h_full, *_ = train_evaluate(holdout_split, feat_cols)      # price + text + news
res_h_base, daily_h_base, *_ = train_evaluate(holdout_split, price_cols)     # price only

def nw_test(s):
    s = pd.Series(s).dropna()
    m = sm.OLS(s.values, np.ones(len(s))).fit(cov_type='HAC', cov_kwds={'maxlags': 20})
    return m.params[0], m.tvalues[0], m.pvalues[0]

rows = []
for name, d in [('price + text + news', daily_h_full), ('price only', daily_h_base),
                ('difference', (daily_h_full - daily_h_base).dropna())]:
    ic, t, p = nw_test(d)
    rows.append({'model': name, 'daily_IC': ic, 't': t, 'p': p, 'days': len(d)})
holdout_results = pd.DataFrame(rows)
print(holdout_results.round(4).to_string(index=False))
holdout_results.to_csv('../../../outputs/v3/holdout_2026.csv', index=False)